## Texas Super Tuesday

In [406]:
import pandas as pd
import numpy as np
from rapidfuzz import process, fuzz
import subprocess
import json

In [407]:
roddy = pd.read_csv('raw_data/roddyoriginals.csv')

roddy.to_csv("export_data/roddy_import.csv", index=False)

In [408]:
#Create full address for record name

roddy["Last Name"] = (
    roddy["Street Number"].astype("string").str.replace(r"\.0$", "", regex=True) + " " +
    roddy["Address"].str.title() + ", " +
    roddy["City Name"].str.title() + ", TX " +
    roddy["Zip"].astype(str).str.replace(r"\.0$", "", regex=True)
)

In [409]:
#Find County

texascounties = pd.read_csv('sf_data/texascounties.csv')

roddy['County'] = roddy['County Code'].map(
    texascounties.drop_duplicates(subset='County Code')
    .set_index('County Code')['County']
)

In [410]:
#Find Market

roddy['Market'] = roddy['County Code'].map(
    texascounties.drop_duplicates(subset='County Code')
    .set_index('County Code')['Market']
)

In [411]:
#Find Lead Manager Name from County Code

roddy['Lead Manager Name'] = roddy['County Code'].map(
    texascounties.drop_duplicates(subset='County Code')
    .set_index('County Code')['Lead Manager']
)


#Find Salesforce ID from Lead Manager Name

leadmanagers = pd.read_csv('sf_data/leadmanagers.csv')

roddy['Lead Manager'] = roddy['Lead Manager Name'].map(
    leadmanagers.drop_duplicates(subset='Lead Manager')
    .set_index('Lead Manager')['Salesforce ID']
)

In [412]:
#Create unique identifier

market_map = {
    'TX Small County': 'TXSMALLCOUNTY',
    'Austin': 'AUSTIN',
    'Waco': 'WACO',
    'Dallas': 'DAL',
    'San Antonio': 'SAN_ANT',
    'Houston': 'HOUS'
}

roddy['APN'] = (
    roddy['Market'].map(market_map)
    + '_' + roddy['Street Number'].astype('string').str.replace(r'\.0$', '', regex=True)
    + '_' + roddy['Address'].astype('string').str.upper().str.replace(' ', '_', regex=False)
    + '_' + roddy['Zip'].astype('string').str.replace(r'\.0$', '', regex=True)
)

In [413]:
#Format County

roddy['County'] = roddy['County'].apply(
    lambda x: x + ' - TX' if pd.notna(x) and x != '' else ''
)

In [414]:
#Add Auction Status

roddy['Auction Status'] = roddy['Class'].apply(
    lambda x: 'Pending' if x == 'Single Family Residential' else 'Killed'
)

In [415]:
#Format Type of Property

roddy['Type of Property'] = roddy['Class'].replace({
    'Single Family Residential': 'SFR'
})

In [416]:
#Format Beds and Baths

roddy['Beds'] = roddy['B/B/G'].astype('string').str.split('/').str[0]
roddy['Baths'] = roddy['B/B/G'].astype('string').str.split('/').str[1]

In [417]:
#Format Pool

roddy = roddy.rename(columns={'Pool': 'notPool'})

roddy['Pool'] = roddy['notPool'].apply(
    lambda x: 'Yes' if x == 'P' else 'No'
)

In [418]:
#Format Owner

roddy['Property Owner Name'] = (
    roddy['Mortgagor First Name'].fillna('').astype(str).str.strip()
    + ' ' +
    roddy['Mortgagor Last Name'].fillna('').astype(str).str.strip()
).str.strip()

In [419]:
#Add Auction Date

roddy['Auction Date Updated'] = '11/3/2026'

In [420]:
#Add Auction Trustee Lookup

roddy['Auction Trustee (Lookup)'] = 'Lookup Needed'

In [421]:
#Add State

roddy['State'] = 'Texas'

In [422]:
#Rename Columns

roddy = roddy.rename(columns={
    'Report ID': 'Roddy Report ID',
    'Loan Type': 'Foreclosing Instrument',
    'Property Sq Footage': 'Bldg Sq Ft',
    'Legal Description 1': 'Legal Description',
    'Mortgagee': 'Beneficiary',
    'Trustee': 'Auction Trustee',
    'Volume': 'Foreclosing Instrument No.',
    'Original Loan Amount': 'Mortgage $',
    'Year Of Construction': 'Year Built',
    'Assessed Value': 'RPR',
    'Status': 'notStatus'
})

In [423]:
roddy[['County', 'Market', 'Last Name', 'APN', 'Lead Manager Name']].head()

,County,Market,Last Name,APN,Lead Manager Name
0,Dallas - TX,Dallas,"4505 Indian Creek Dr, Balch Springs, TX 75180",DAL_4505_INDIAN_CREEK_DR_75180,Brian Glasser
1,Dallas - TX,Dallas,"14825 Magnolia Ln, Balch Springs, TX 75180",DAL_14825_MAGNOLIA_LN_75180,Brian Glasser
2,Dallas - TX,Dallas,"14629 Oakwood Ln, Balch Springs, TX 75180",DAL_14629_OAKWOOD_LN_75180,Brian Glasser
3,Dallas - TX,Dallas,"12101 Bishop Dr, Balch Springs, TX 75180",DAL_12101_BISHOP_DR_75180,Brian Glasser
4,Dallas - TX,Dallas,"11816 Sunrise Dr, Balch Springs, TX 75180",DAL_11816_SUNRISE_DR_75180,Brian Glasser


## Match Addresses

In [424]:
query = """
SELECT
    Id,
    Name,
    Auction_Status__c,
    Auction_Trustee_Lookup__r.Name,
    Auction_Listing_ID__c,
    Auction_Opening_Bid__c
FROM Lead
WHERE (
    Auction_Date__c = LAST_N_DAYS:60
    OR Auction_Date__c = NEXT_N_DAYS:30)
AND State_Lookup__r.Name = 'Texas'
"""

result = subprocess.run(
    [
        "sf", "data", "query",
        "--query", query,
        "--target-org", "production_org",
        "--json"
    ],
    capture_output=True,
    text=True,
    env={**os.environ, "NO_COLOR": "1", "FORCE_COLOR": "0"}
)

if result.returncode != 0:
    print("Salesforce Error:")
    print(result.stderr)
    print(result.stdout[:1000])

else:
    # Remove ANSI color codes if any remain
    import re

    clean_output = re.sub(
        r'\x1b\[[0-9;]*m',
        '',
        result.stdout
    )

    data = json.loads(clean_output)

    sf_leads = pd.json_normalize(data['result']['records'])

    print(f"Successfully retrieved {len(sf_leads)} Salesforce records")
    display(sf_leads.head())

Successfully retrieved 5437 Salesforce records


,Id,Name,Auction_Status__c,Auction_Listing_ID__c,Auction_Opening_Bid__c,attributes.type,attributes.url,Auction_Trustee_Lookup__r.attributes.type,Auction_Trustee_Lookup__r.attributes.url,Auction_Trustee_Lookup__r.Name
0,00QQg00000h8lfOMAQ,"8048 Caladium Dr, North Richland Hills, TX 76180",Cancelled,2129172,160000.0,Lead,/services/data/v68.0/sobjects/Lead/00QQg00000h...,Contact__c,/services/data/v68.0/sobjects/Contact__c/a1fQg...,Auction.com
1,00QQg00000h8lfYMAQ,"5667 Westhaven Dr, Fort Worth, TX 76132",Pending,None,NaN,Lead,/services/data/v68.0/sobjects/Lead/00QQg00000h...,Contact__c,/services/data/v68.0/sobjects/Contact__c/a1fQg...,Lookup Needed
2,00QQg00000h8lfiMAA,"8364 Meadow Sweet Ln, Fort Worth, TX 76123",Active,None,264120.0,Lead,/services/data/v68.0/sobjects/Lead/00QQg00000h...,Contact__c,/services/data/v68.0/sobjects/Contact__c/a1fQg...,Lookup Needed
3,00QQg00000h946xMAA,"3211 Kellie St, Sachse, TX 75048",Cancelled,None,NaN,Lead,/services/data/v68.0/sobjects/Lead/00QQg00000h...,Contact__c,/services/data/v68.0/sobjects/Contact__c/a1fQg...,Xome.com
4,00QQg00000h9476MAA,"7218 Bursey Rd, North Richland Hills, TX 76182",Pending,None,NaN,Lead,/services/data/v68.0/sobjects/Lead/00QQg00000h...,Contact__c,/services/data/v68.0/sobjects/Contact__c/a1fQg...,Lookup Needed


In [425]:
#Rename sf columns 

sf_leads = sf_leads.rename(columns={
    'Name': 'Last Name',
    'Auction_Status__c': 'Auction Status',
    'Auction_Trustee_Lookup__r.Name': 'Auction Trustee',
    'Auction_Listing_ID__c': 'Auction Listing ID',
    'Auction_Opening_Bid__c': 'Auction Opening Bid'
})

In [426]:
sf_leads[['Last Name', 'Auction Status', 'Auction Trustee',
          'Auction Listing ID', 'Auction Opening Bid']].head()

,Last Name,Auction Status,Auction Trustee,Auction Listing ID,Auction Opening Bid
0,"8048 Caladium Dr, North Richland Hills, TX 76180",Cancelled,Auction.com,2129172,160000.0
1,"5667 Westhaven Dr, Fort Worth, TX 76132",Pending,Lookup Needed,None,NaN
2,"8364 Meadow Sweet Ln, Fort Worth, TX 76123",Active,Lookup Needed,None,264120.0
3,"3211 Kellie St, Sachse, TX 75048",Cancelled,Xome.com,None,NaN
4,"7218 Bursey Rd, North Richland Hills, TX 76182",Pending,Lookup Needed,None,NaN


In [427]:
#Fuzzy Match

threshold = 85

# Normalize names
def normalize_name(value):
    if pd.isna(value):
        return ""
    return " ".join(str(value).upper().strip().split())

# Salesforce lookup
sf_lookup = {}

for name in sf_leads['Last Name'].dropna().unique():
    normalized = normalize_name(name)
    if normalized:
        sf_lookup.setdefault(normalized, name)

sf_names = list(sf_lookup.keys())

# Keep original Roddy Last Name
roddy['Roddy Last Name'] = roddy['Last Name']

# Fuzzy match
def fuzzy_match_name(name):
    normalized = normalize_name(name)

    if not normalized or not sf_names:
        return pd.Series([None, None])

    match = process.extractOne(
        normalized,
        sf_names,
        scorer=fuzz.ratio
    )

    if match:
        matched_name, score, _ = match
        return pd.Series([sf_lookup[matched_name], round(score, 2)])

    return pd.Series([None, None])

# Store Salesforce match and confidence
roddy[['Salesforce Last Name', 'Match Confidence']] = (
    roddy['Roddy Last Name'].apply(fuzzy_match_name)
)

# Use Salesforce Last Name only if confidence >= 85
roddy['Last Name'] = roddy['Roddy Last Name']

matched = roddy['Match Confidence'] >= threshold

roddy.loc[matched, 'Last Name'] = (
    roddy.loc[matched, 'Salesforce Last Name']
)

In [428]:
#View matches

display(roddy[
    ['Roddy Last Name', 'Salesforce Last Name', 'Match Confidence']
])

,Roddy Last Name,Salesforce Last Name,Match Confidence
0,"4505 Indian Creek Dr, Balch Springs, TX 75180","4505 Indian Creek Dr, Balch Springs, TX 75180",100.0
1,"14825 Magnolia Ln, Balch Springs, TX 75180","14825 Magnolia Ln, Balch Springs, TX 75180",100.0
2,"14629 Oakwood Ln, Balch Springs, TX 75180","14629 Oakwood Ln, Balch Springs, TX 75180",100.0
3,"12101 Bishop Dr, Balch Springs, TX 75180","12101 Bishop Dr, Balch Springs, TX 75180",100.0
4,"11816 Sunrise Dr, Balch Springs, TX 75180","11816 Sunrise Dr, Balch Springs, TX 75180",100.0
...,...,...,...
255,"2903 Balleywood Dr, Seagoville, TX 75159","2903 Balleywood Dr, Seagoville, TX 75159",100.0
256,"3009 Melrose Dr, Seagoville, TX 75159","3009 Melrose Dr, Seagoville, TX 75159",100.0
257,"1930 Wooley Way, Seagoville, TX 75159","1930 Wooley Way, Seagoville, TX 75159",100.0
258,"353 Tavenner Ln, Sunnyvale, TX 75182","353 Tavenner Ln, Sunnyvale, TX 75182",100.0


In [429]:
#Look at differences

#Common street suffixes
suffix_map = {
    'DRIVE': 'DR',
    'STREET': 'ST',
    'AVENUE': 'AVE',
    'ROAD': 'RD',
    'LANE': 'LN',
    'BOULEVARD': 'BLVD',
    'COURT': 'CT',
    'CIRCLE': 'CIR',
    'PLACE': 'PL',
    'TERRACE': 'TER',
    'TRAIL': 'TRL',
    'PARKWAY': 'PKWY',
    'HIGHWAY': 'HWY',
    'WAY': 'WAY'
}

def get_address_parts(address):
    if pd.isna(address):
        return '', ''

    address = str(address).strip()
    parts = [p.strip() for p in address.split(',')]

    # Expected format: Street, City, TX ZIP
    street = parts[0] if len(parts) >= 1 else ''
    city = parts[1] if len(parts) >= 3 else ''

    return street.upper(), city.upper()

def normalize_street(street):
    street = re.sub(r'[^\w\s]', '', street.upper())
    words = street.split()
    return ' '.join(suffix_map.get(word, word) for word in words)

def compare_addresses(row):
    roddy_street, roddy_city = get_address_parts(row['Roddy Last Name'])
    sf_street, sf_city = get_address_parts(row['Salesforce Last Name'])

    # Detect abbreviation-only differences
    suffix_difference = (
        roddy_street != sf_street
        and normalize_street(roddy_street) == normalize_street(sf_street)
    )

    city_difference = roddy_city != sf_city

    return pd.Series({
        'Street Suffix Difference': suffix_difference,
        'City Difference': city_difference,
        'Roddy City': roddy_city,
        'Salesforce City': sf_city
    })

# Compare matched records
address_comparison = roddy[
    (roddy['Match Confidence'] >= 85) &
    roddy['Salesforce Last Name'].notna()
].copy()

address_comparison = address_comparison.join(
    address_comparison.apply(compare_addresses, axis=1)
)

In [430]:
differences = address_comparison[
    address_comparison['Street Suffix Difference'] |
    address_comparison['City Difference']
]

display(differences[
    [
        'Roddy Last Name',
        'Salesforce Last Name',
        'Match Confidence',
        'Street Suffix Difference',
        'City Difference'
    ]
])

,Roddy Last Name,Salesforce Last Name,Match Confidence,Street Suffix Difference,City Difference


In [431]:
print("Street suffix differences:",
      address_comparison['Street Suffix Difference'].sum())

print("City differences:",
      address_comparison['City Difference'].sum())

print("Both:",
      (
          address_comparison['Street Suffix Difference'] &
          address_comparison['City Difference']
      ).sum())

Street suffix differences: 0
City differences: 0
Both: 0


In [432]:
low_confidence = roddy[
    (roddy['Match Confidence'] < 85) |
    (roddy['Match Confidence'].isna())
]

display(low_confidence[
    ['Last Name', 'Match Confidence']
])

,Last Name,Match Confidence


In [433]:
#Update Lead Manager Ernie vs Brian

# Create Salesforce lookup using Last Name
sf_lookup = (
    sf_leads.drop_duplicates(subset='Last Name')
    .set_index('Last Name')
)

# Get Auction Trustee from matched Salesforce record
roddy['Salesforce Auction Trustee'] = (
    roddy['Salesforce Last Name'].map(
        sf_lookup['Auction Trustee']
    )
)

# Identify existing Salesforce properties with Auction.com
auction_com_match = (
    (roddy['Match Confidence'] >= 85) &
    (roddy['Salesforce Auction Trustee'].str.strip().str.casefold() == 'auction.com')
)

# Update Lead Manager Name
roddy.loc[auction_com_match, 'Lead Manager Name'] = 'Ernie Loreto'

# Find Ernie Loreto's Salesforce ID
ernie_id = (
    leadmanagers.loc[
        leadmanagers['Lead Manager'].str.strip().str.casefold() == 'ernie loreto',
        'Salesforce ID'
    ].dropna()
)

if ernie_id.empty:
    raise ValueError("Ernie Loreto was not found in leadmanagers.csv")

# Update Lead Manager Salesforce ID
roddy.loc[auction_com_match, 'Lead Manager'] = ernie_id.iloc[0]

In [434]:
display(
    roddy.loc[
        auction_com_match,
        [
            'Last Name',
            'Salesforce Last Name',
            'Match Confidence',
            'Salesforce Auction Trustee',
            'Lead Manager Name',
            'Lead Manager'
        ]
    ]
)

print(f"Properties assigned to Ernie Loreto: {auction_com_match.sum()}")

,Last Name,Salesforce Last Name,Match Confidence,Salesforce Auction Trustee,Lead Manager Name,Lead Manager
0,"4505 Indian Creek Dr, Balch Springs, TX 75180","4505 Indian Creek Dr, Balch Springs, TX 75180",100.0,Auction.com,Ernie Loreto,005Qg00000UWrpV
2,"14629 Oakwood Ln, Balch Springs, TX 75180","14629 Oakwood Ln, Balch Springs, TX 75180",100.0,Auction.com,Ernie Loreto,005Qg00000UWrpV
14,"745 Lovern St, Cedar Hill, TX 75104","745 Lovern St, Cedar Hill, TX 75104",100.0,Auction.com,Ernie Loreto,005Qg00000UWrpV
18,"1107 Germany Dr, Cedar Hill, TX 75104","1107 Germany Dr, Cedar Hill, TX 75104",100.0,Auction.com,Ernie Loreto,005Qg00000UWrpV
21,"652 Deforest Rd, Coppell, TX 75019","652 Deforest Rd, Coppell, TX 75019",100.0,Auction.com,Ernie Loreto,005Qg00000UWrpV
...,...,...,...,...,...,...
239,"102 N Waterview Dr, Richardson, TX 75080","102 N Waterview Dr, Richardson, TX 75080",100.0,Auction.com,Ernie Loreto,005Qg00000UWrpV
240,"1911 Harvard Dr, Richardson, TX 75081","1911 Harvard Dr, Richardson, TX 75081",100.0,Auction.com,Ernie Loreto,005Qg00000UWrpV
245,"6702 Lakeshore Dr, Rowlett, TX 75089","6702 Lakeshore Dr, Rowlett, TX 75089",100.0,Auction.com,Ernie Loreto,005Qg00000UWrpV
255,"2903 Balleywood Dr, Seagoville, TX 75159","2903 Balleywood Dr, Seagoville, TX 75159",100.0,Auction.com,Ernie Loreto,005Qg00000UWrpV


Properties assigned to Ernie Loreto: 84


In [435]:
roddy.to_csv("export_data/roddy_import.csv", index=False)